# Расчёт систолического напряжения стенки левого желудочка (Grossman formula)

Этот ноутбук содержит код для расчёта **систолического напряжения стенки левого желудочка (г/см²)** и его интеграции в датасет `final_analysis_dataset_ver2.xlsx`.

### Формула Гроссмана:
$$\sigma = 0.334 \times \frac{P \times d}{h \times (1 + \frac{h}{d})}$$
Где:
- $P$ — систолическое давление (`exercise_peak_systolic_bp_mmhg` / САД НАГР)
- $d$ — конечно-систолический размер ЛЖ (`echo_lv_end_systolic_diameter_mm` / КСР)
- $h$ — толщина задней стенки левого желудочка в систолу (`echo_lv_posterior_wall_thickness_mm` / ЗС)

### Важный научный нюанс:
В исходном датасете столбец `echo_lv_posterior_wall_thickness_mm` (ЗС) заполнен только для 15 пациентов. При этом эти 15 пациентов **не пересекаются** с 66 пациентами, прошедшими нагрузочные тесты (для которых заполнено давление `exercise_peak_systolic_bp_mmhg`). Из-за этого прямой расчёт «в лоб» даёт 0 заполненных значений.

Тем не менее, в датасете присутствует расчетный показатель относительной толщины стенки — `echo_lv_relative_wall_thickness_ratio` (ОТС / RWT), который рассчитывается по формуле:
$$RWT = \frac{2 \times LVPW}{LVEDd}$$
Это позволяет нам **математически восстановить** толщину задней стенки ($LVPW$ / $h$) для остальных пациентов, у которых заполнены RWT и LVEDd:
$$h = \frac{RWT \times LVEDd}{2}$$

Благодаря этому методу мы получаем **64 успешно рассчитанных значения** систолического напряжения стенки ЛЖ!

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Пути к файлам
DATASET_PATH = Path('../data/final_analysis_dataset_ver2.xlsx')
if not DATASET_PATH.exists():
    DATASET_PATH = Path('data/final_analysis_dataset_ver2.xlsx')

In [2]:
# Загрузка датасета
df = pd.read_excel(DATASET_PATH)
print("Размерность датасета до добавления колонки:", df.shape)
print("Количество непустых значений до восстановления:")
print("  САД нагрузки (exercise_peak_systolic_bp_mmhg):", df['exercise_peak_systolic_bp_mmhg'].notna().sum())
print("  КСР (echo_lv_end_systolic_diameter_mm):       ", df['echo_lv_end_systolic_diameter_mm'].notna().sum())
print("  ЗС (echo_lv_posterior_wall_thickness_mm):     ", df['echo_lv_posterior_wall_thickness_mm'].notna().sum())
print("  ОТС (echo_lv_relative_wall_thickness_ratio):  ", df['echo_lv_relative_wall_thickness_ratio'].notna().sum())
print("  КДР (echo_lv_end_diastolic_diameter_mm):      ", df['echo_lv_end_diastolic_diameter_mm'].notna().sum())

Размерность датасета до добавления колонки: (153, 215)
Количество непустых значений до восстановления:
  САД нагрузки (exercise_peak_systolic_bp_mmhg): 66
  КСР (echo_lv_end_systolic_diameter_mm):        89
  ЗС (echo_lv_posterior_wall_thickness_mm):      15
  ОТС (echo_lv_relative_wall_thickness_ratio):   80
  КДР (echo_lv_end_diastolic_diameter_mm):       89


In [3]:
# Восстановление толщины задней стенки (h) через RWT и LVEDd для пропущенных значений
reconstructed_h = df['echo_lv_relative_wall_thickness_ratio'] * df['echo_lv_end_diastolic_diameter_mm'] / 2

# Заполняем пропуски в ЗС восстановленными значениями
df['echo_lv_posterior_wall_thickness_mm'] = df['echo_lv_posterior_wall_thickness_mm'].fillna(reconstructed_h)
print("Количество непустых значений ЗС после восстановления:", df['echo_lv_posterior_wall_thickness_mm'].notna().sum())

Количество непустых значений ЗС после восстановления: 89


In [4]:
# Выделение переменных для расчёта
P = df['exercise_peak_systolic_bp_mmhg']
d = df['echo_lv_end_systolic_diameter_mm']
h = df['echo_lv_posterior_wall_thickness_mm']

# Расчёт систолического напряжения стенки ЛЖ по формуле Гроссмана
df['echo_lv_systolic_wall_stress_g_cm2'] = 0.334 * P * d / (h * (1 + (h / d)))

In [5]:
# Оценка результатов
print("Размерность датасета после расчёта:", df.shape)
print("Количество успешно рассчитанных значений напряжения стенки:", df['echo_lv_systolic_wall_stress_g_cm2'].notna().sum())
print("\nОписательная статистика новой переменной:")
print(df['echo_lv_systolic_wall_stress_g_cm2'].describe())
print("\nПервые 10 строк с рассчитанными значениями:")
cols_show = ['subject_full_name', 'exercise_peak_systolic_bp_mmhg', 'echo_lv_end_systolic_diameter_mm', 'echo_lv_posterior_wall_thickness_mm', 'echo_lv_systolic_wall_stress_g_cm2']
print(df[cols_show].dropna().head(10))

Размерность датасета после расчёта: (153, 215)
Количество успешно рассчитанных значений напряжения стенки: 64

Описательная статистика новой переменной:
count     64.000000
mean     116.411474
std       70.681943
min       54.346357
25%       80.911240
50%       95.881231
75%      117.904672
max      451.937339
Name: echo_lv_systolic_wall_stress_g_cm2, dtype: float64

Первые 10 строк с рассчитанными значениями:
   subject_full_name  exercise_peak_systolic_bp_mmhg  \
0          Абатурова                           159.0   
2            Айтиева                           161.0   
3             Алиева                           155.0   
4           Артемова                           139.0   
5           Бирюкова                           147.0   
6           Бузакова                           140.0   
7           Булатова                           148.0   
8          Варвашеня                           149.0   
9            Волкова                           147.0   
10         Гагарская     

In [6]:
# Сохранение датасета
df.to_excel(DATASET_PATH, index=False)
print("Обновлённый датасет успешно сохранён по пути:", DATASET_PATH)

Обновлённый датасет успешно сохранён по пути: ..\data\final_analysis_dataset_ver2.xlsx
